# Deliverable 2

### Amalie K. Meltvedt and Maren J. Borgaas

### Date: 

## GitHub Repository Link
The GitHub repository can be accessed through this link: https://github.com/marenjb/Deliverable-2.git

## Importing the necessary libraries

In [ ]:
# Adding necessary packages
from pathlib import Path
import re
import json

## Task 1
To create the file reader function, we used the `pathlib.Path` function to handle file paths. In order for it to work on both UTF-8 and ISO-8859-1/Latin-1 encoding, we used a try-except statement where it will first test UTF-8, then  ISO-8859-1. The appropriate encoding will then be stored in the `encoding_used` variable. This function will work when the variable `file_path` has been given the appropiate path.

In [ ]:
# Using a try-except statement to create the file reader function
def read_file(file_path: Path) -> tuple[str,str]:
    """Read the files and return the text content and the encoding used"""
    try:
        file_text_content = file_path.read_text(encoding = "utf-8")
        encoding_used = "utf-8"
    except:
        file_text_content = file.path.read_text(encoding = "ISO-8859-1")
        encoding_used = "ISO-8859-1"
    
    return file_text_content, encoding_used

Next, we created a log parser function using the `re` module. The regular expression pattern is stored in the `pattern` variable, meaning the function will recognize lines that use the same pattern. `pattern.fullmatch()` and `if not match: ignore` ensures only exact matches are added, meaning noise lines will be ignored. It will then capture the key fields `timestamp`, `register_id`, `item`, `price`, `currency` and `status`, save them in dictionaries, and finally add the dictionaries in the `records` list.

In [ ]:
# Creating the log parser function
def parse_log(file_text_content: str) -> list[dict]:
    """ Parse the log text and return the valid transaction records"""
    
    # Creating an empty list for the records
    records = []
    
    # Storing the regex pattern
    pattern = re.compile(r"\[(.*?)\] \[(.*?)\] item=(.*?) price=(.*?) (.*?) \((.*?)\)")
    
    # Going through each line and storing matching expressions in a match variable
    for line in file_text_content.splitlines(): # Splitlines to go through each line
        match = pattern.fullmatch(line)
        
        if not match:
            continue
        
        # Collecting the key fields
        timestamp = match.group(1)
        register_id = match.group(2)
        item = match.group(3)
        price = match.group(4)
        currency = match.group(5)
        status = match.group(6)
        
        # Saving the key fields data in a dictionary
        record = {
            "timestamp": timestamp,
            "register_id": register_id,
            "item": item,
            "price": price,
            "currency": currency,
            "status": status,
        }
        
        # Appending the data to the records list
        records.append(record)
        
        return records

## Task 2

To create a recursive directory search function that finds all log files, we used `Path.iterdir()` to iterate through the items. We then used `.is_dir()` to check wether an item is a directory. If it is, the function will call itself recursively to search through the subdirectory. Because the recursive function call returns a list, we used `.extend()` to add each element from the returned list to `log_files` individually. If an item is a file, the code will check if it has a `.log` extension using `.suffix`, and then add it to the list.

In [ ]:
# Creating the log file finder function
def find_log_files(directory: str | Path) -> list[Path]:
    """Find all log files in a directory and in all subdirectories"""

    # Converting the directory to a Path object
    directory = Path(directory)
    
    # Creating an empty list to store the log files
    log_files = []

    # Iterating through the directory and its subdirectories
    for item in directory.iterdir():

        # Checking if the item is a directory, and using recursion if it is, adds the files to the list
        if item.is_dir():
            log_files.extend(find_log_files(item))

        # Checking if item is a file and if it has .log extension, adds it to the list
        elif item.is_file() and item.suffix == ".log":
            log_files.append(item)

    return log_files

The function below is designed to traverse any combination of nested dictionaries and lists in `store_config.json`. The code uses `isinstance()` to check the datatype. For example, we first check whether data is a dictionary. We then use `.get()` to retrieve the values associated with the keys "active" and "id". If the register is active and its ID is a string, the ID is added to the set using `.add()`. To search through nested dictionaries and lists, the function uses recursion and `.update()` to add active registers to the set. A `set` will return only unique active register IDs.

In [6]:
# Creating the function to find active registers
def find_active_registers(data: object) -> set[str]:
    """Find all register IDs where active is true"""

    # Creating an empty set to store the active register IDs
    active_registers = set()

    # Checing is the data is a dictionary
    if isinstance(data, dict):

        # Get the register ID
        register_id = data.get("id")
        
        # Checking if the dictionary has the value True for the key "active"
        if data.get("active") is True:
            register_id = data.get("id")
        
        # Checking if the register ID is a string and adding it to the set
        if isinstance(register_id, str):
                active_registers.add(register_id)
        
        # Using recursion to go through the values of the dictionary and find active registers 
        for value in data.values():
            active_registers.update(find_active_registers(value))
    
    # Checking if the data is a list
    elif isinstance(data, list):
        
        # Using recursion to go through the items of the list and find active registers
        for item in data:
            active_registers.update(find_active_registers(item))

    return active_registers
